# Cruzamento Espectral — Espectro JONSWAP, RAO e Heave

Cálculo do espectro de energia de ondas (JONSWAP), cruzamento com o RAO de um navio
para obter o espectro de resposta de heave, geração de séries temporais e
verificação via FFT.

## 1. Configuração

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

matplotlib.style.use('ggplot')
matplotlib.rcParams.update({'legend.labelcolor': 'black'})

pi = np.pi

## 2. Espectro de Energia do Mar

Tipos de espectro: Pierson-Moskowitz, Bretschneider, **JONSWAP**,
Ochi-Hubble, Torsethaugen.

### 2.1 Espectro de JONSWAP

$$
S_w = \frac{1}{2\pi} \cdot \frac{5}{16} \cdot h_s^2 \cdot t_p \cdot
\left(\frac{\omega_p}{\omega}\right)^5 \cdot
(1 - 0.287 \ln\gamma) \cdot
\exp\!\left(-1.25 \left(\frac{\omega}{\omega_p}\right)^{-4}\right) \cdot
\gamma^{\exp\!\left(-\dfrac{(\omega-\omega_p)^2}{2\,\sigma^2\,\omega_p^2}\right)}
$$

onde $\sigma = 0.07$ para $\omega \le \omega_p$ e $\sigma = 0.09$ para $\omega > \omega_p$.

In [ ]:
N = 5000
tp = 20
hs = 5

w = np.linspace(2 * pi / 100, 2 * pi / 2, N)

In [ ]:
def jonswap(hs, tp):
    gama = 6.4 * tp**-0.491
    omega_p = 2 * pi / tp
    # σ = 0.07 for ω ≤ ωp, σ = 0.09 for ω > ωp
    sigma = (w <= omega_p) * 0.07 + (w > omega_p) * 0.09
    exponent = -((w - omega_p) / (2 * pi)) ** 2 / (
        2 * sigma**2 * (omega_p / (2 * pi)) ** 2
    )
    return (
        (2 * pi) ** -1
        * (5 / 16)
        * hs**2
        * tp
        * (omega_p / w) ** 5
        * (1 - 0.287 * np.log(gama))
        * np.exp(-1.25 * (w / omega_p) ** -4)
        * gama ** np.exp(exponent)
    )

In [ ]:
Sw = jonswap(hs, tp)

plt.figure(figsize=(12, 6))
plt.plot(2 * pi / w, Sw, linewidth=2, label=f'JONSWAP — Hs={hs} m, Tp={tp} s')
plt.axvline(x=tp, color='red', linestyle='--', alpha=0.7)
plt.ylabel('Densidade Espectral S(ω) [m²·s]', fontsize=12)
plt.xlabel('Período de Onda T [s]', fontsize=12)
plt.title('Espectro de Energia de Ondas — Modelo JONSWAP', fontsize=14)
plt.grid(True, alpha=0.7)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
idx_pico = np.argmax(Sw)
T_pico = 2 * pi / w[idx_pico]
S_max = Sw[idx_pico]
gama = 6.4 * tp**-0.491

m0 = np.trapezoid(Sw, w)
Hs_calculado = 4 * np.sqrt(m0)

print(f'Fator de pico γ:   {gama:.3f}')
print(f'Período no pico:   {T_pico:.2f} s')
print(f'S máximo:          {S_max:.4f} m²·s')
print(f'Hs calculado:      {Hs_calculado:.3f} m')

In [ ]:
m0_teo = np.trapezoid(Sw, w)
m2_teo = np.trapezoid(Sw * (w**2), w)

Hs_teo = 4 * np.sqrt(m0_teo)
Tz_teo = 2 * np.pi * np.sqrt(m0_teo / m2_teo)

print(f'm0 = {m0_teo:.4f} m²')
print(f'm2 = {m2_teo:.4f} m²/rad²')
print(f'Hs = {Hs_teo:.2f} m')
print(f'Tz = {Tz_teo:.2f} s')

### 2.2 Conversão para o Domínio da Frequência (Hz)

Conservação de energia: $S(f)\,\Delta f = S(\omega)\,\Delta\omega$, logo
$S(f) = S(\omega) \cdot 2\pi$.

In [ ]:
Sf = Sw * 2 * pi
f = w / (2 * pi)

delta_f = np.diff(f)[0]
f_pico = 1 / tp

plt.figure(figsize=(12, 6))
plt.plot(f, Sf, linewidth=2.5, label=f'JONSWAP — Hs={hs} m, Tp={tp} s')
plt.axvline(x=f_pico, color='red', linestyle='--', alpha=0.8, linewidth=1.5)
plt.ylabel('Densidade Espectral S(f) [m²·s]', fontsize=12)
plt.xlabel('Frequência f [Hz]', fontsize=12)
plt.title('Espectro de Energia de Ondas — Domínio da Frequência', fontsize=14)
plt.grid(True, alpha=0.6)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
m0_f = np.trapezoid(Sf, f)
Hs_teo_f = 4 * np.sqrt(m0_f)

print(f'Δf:              {delta_f:.6f} Hz')
print(f'Frequência pico: {f_pico:.4f} Hz')
print(f'm0 (domínio f):  {m0_f:.6f} m²')
print(f'Hs (domínio f):  {Hs_teo_f:.4f} m')
Hs_teo_f

### 2.3 Amplitudes Espectrais

Amplitude de cada componente harmônica:

$$\zeta_a(f) = \sqrt{2\,\Delta f\,S(f)}$$

In [ ]:
zeta_a = np.sqrt(2 * delta_f * Sf)

plt.figure(figsize=(12, 6))
plt.plot(f, zeta_a, linewidth=2, label='Amplitudes das Componentes')
plt.axvline(x=f_pico, color='red', linestyle='--', alpha=0.8, linewidth=1.5)
plt.ylabel('Amplitude ζₐ [m]', fontsize=12)
plt.xlabel('Frequência f [Hz]', fontsize=12)
plt.title('Amplitudes das Componentes de Onda', fontsize=14)
plt.grid(True, alpha=0.6)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(f'Amplitude máxima: {np.max(zeta_a):.4f} m em {f[np.argmax(zeta_a)]:.4f} Hz')
print(f'Amplitude média:  {np.mean(zeta_a):.4f} m')

## 3. RAO do Navio

O **Response Amplitude Operator (RAO)** é a função de transferência que relaciona
a amplitude de resposta da estrutura com a amplitude das ondas incidentes.

- $|\text{RAO}| \approx 1$: estrutura acompanha as ondas;
- $|\text{RAO}| \gg 1$: ressonância;
- $|\text{RAO}| \ll 1$: estrutura pouco afetada por aquela frequência.

In [ ]:
rao_data = np.array([
    [0.00999999253114992, 1.000626, -0.001304847],
    [0.0499998830785236, 1.015586, -0.007924744],
    [0.0999999253114992, 1.065954, -0.0256068],
    [0.150000007333373, 1.159031, -0.07442564],
    [0.175000078185924, 1.221892, -0.1229897],
    [0.200000168932916, 1.292448, -0.1973969],
    [0.224999742426387, 1.364457, -0.3068606],
    [0.250000410110318, 1.425854, -0.4621112],
    [0.275000560540776, 1.4532, -0.6728624],
    [0.299999298471141, 1.40566, -0.9354835],
    [0.324999627949226, 1.236216, -1.208878],
    [0.349999181549665, 0.9352049, -1.407357],
    [0.374999123088927, 0.5715443, -1.451135],
    [0.399999064628189, 0.255577, -1.342692],
    [0.42499900616745, 0.04731597, -1.157613],
    [0.450001096298654, -0.06280198, -0.9670042],
    [0.474998511255053, -0.1095895, -0.8031601],
    [0.499998830785236, -0.1219991, -0.6717551],
    [0.524998772324498, -0.1177207, -0.5685172],
    [0.549998713863759, -0.1063931, -0.487183],
    [0.574998884187273, -0.09290343, -0.4220923],
    [0.599998596942283, -0.07947022, -0.3687671],
    [0.624999781876196, -0.06680158, -0.3239916],
    [0.649999928327242, -0.05496959, -0.2856781],
    [0.675000194144612, -0.04388655, -0.25251],
    [0.699999922813953, -0.03351789, -0.2236753],
    [0.725000208525694, -0.02390925, -0.1985342],
    [0.750000036666864, -0.01515082, -0.1766077],
    [0.774999606181508, -0.007343339, -0.1574261],
    [0.800000166435309, -0.0005663104, -0.1405595],
    [0.825000237287859, 0.005157016, -0.1255997],
    [0.850000312119295, 0.009882366, -0.1121679],
    [0.875000390929618, 0.01373685, -0.09997757],
    [0.899999614282054, 0.01686178, -0.08883087],
    [0.925000376463842, 0.01938278, -0.07862416],
    [0.949999895247508, 0.02138249, -0.06928597],
    [0.975000396813239, 0.02289039, -0.06080105],
    [0.999999253114992, 0.02392741, -0.05309147],
    [1.02499947914328, 0.02451225, -0.04607917],
    [1.04999929932697, 0.0246883, -0.03968797],
    [1.0750006513767, 0.0245215, -0.03386452],
    [1.09999935349669, 0.02406415, -0.02857455],
    [1.12500072643568, 0.02336604, -0.02380708],
    [1.14999987319435, 0.02245245, -0.0195329],
    [1.17500038470723, 0.02135502, -0.0157248],
    [1.1999994857094, 0.02011512, -0.01235669],
    [1.22500020611285, 0.01876117, -0.009397379],
    [1.24999956375239, 0.01733484, -0.006831634],
    [1.27499960575804, 0.01585459, -0.00462771],
    [1.29999985665448, 0.01435427, -0.002757248],
    [1.32499906309736, 0.01286535, -0.001194537],
    [1.35000038828922, 0.01141375, 7.887054e-05],
    [1.37499979367505, 0.010009, 0.001085244],
    [1.39999984562791, 0.008665669, 0.001860468],
    [1.42500092015186, 0.007401119, 0.002439321],
    [1.45000041705139, 0.006231181, 0.002842027],
    [1.47499883965632, 0.005160025, 0.003085382],
    [1.50000007333373, 0.004188926, 0.003197805],
    [1.52500055997874, 0.003317361, 0.003204818],
    [1.54999921236302, 0.002551197, 0.003131752],
])

In [ ]:
w_rao = rao_data[:, 0]
rao_complex = rao_data[:, 1] + 1j * rao_data[:, 2]

plt.figure(figsize=(10, 6))
plt.plot(w_rao, np.abs(rao_complex), 'x-', linewidth=1.5, markersize=6, label='|RAO(ω)|')
plt.ylabel('RAO(ω) [m/m]', fontsize=12)
plt.xlabel('ω [rad/s]', fontsize=12)
plt.title('Operador de Amplitude de Resposta (RAO)', fontsize=14)
plt.grid(True, alpha=0.3, linestyle='--')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
rao_interpolado = np.interp(w, w_rao, rao_complex)

plt.figure(figsize=(12, 7))
plt.plot(
    w, np.abs(rao_interpolado), '.-',
    linewidth=2, markersize=4, label='RAO Interpolado', color='steelblue', alpha=0.8,
)
plt.plot(
    w_rao, np.abs(rao_complex), 'r+',
    markersize=8, markeredgewidth=2, label='RAO Original',
)
plt.ylabel('RAO(ω) [m/m]', fontsize=12)
plt.xlabel('ω [rad/s]', fontsize=12)
plt.title('Comparação: RAO Original vs. Interpolado', fontsize=14)
plt.grid(True, alpha=0.3, linestyle='--')
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

## 4. Espectro de Resposta de Heave

Descreve como a energia do movimento vertical da estrutura está distribuída em
frequência:

$$S_{\text{heave}}(\omega) = |\text{RAO}(\omega)|^2 \cdot S_{\text{mar}}(\omega)$$

In [ ]:
S_heave_w = (np.abs(rao_interpolado) ** 2) * Sw

plt.figure(figsize=(12, 7))
plt.plot(w, S_heave_w, linewidth=2, color='darkblue', label='S_heave(ω)')
plt.fill_between(w, S_heave_w, alpha=0.3, color='skyblue')
plt.ylabel('S_heave(ω) [m²·s]', fontsize=12)
plt.xlabel('ω [rad/s]', fontsize=12)
plt.title('Espectro de Densidade de Energia — Movimento de Heave', fontsize=14)
plt.grid(True, alpha=0.3, linestyle='--')
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
variancia_heave = np.trapezoid(S_heave_w, w)
desvio_padrao_heave = np.sqrt(variancia_heave)
amplitude_significativa = 4 * desvio_padrao_heave

print(f'Variância do heave:       {variancia_heave:.4f} m²')
print(f'Desvio padrão (RMS):      {desvio_padrao_heave:.4f} m')
print(f'Amplitude significativa:  {amplitude_significativa:.4f} m')

In [ ]:
S_heave_f = S_heave_w * (2 * np.pi)
zeta_heave = np.sqrt(2 * delta_f * S_heave_f)

plt.figure(figsize=(12, 7))
plt.plot(f, np.abs(zeta_heave), '.', markersize=6, color='darkgreen', alpha=0.7)
plt.plot(f, np.abs(zeta_heave), '-', linewidth=0.5, color='green', alpha=0.3)
plt.ylabel('ζₐ [m]', fontsize=12)
plt.xlabel('f [Hz]', fontsize=12)
plt.title('Amplitudes das Componentes Harmônicas do Movimento de Heave', fontsize=14)
plt.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.show()

## 5. Séries Temporais

Decomposição em componentes harmônicas independentes com fases aleatórias:

$$y(t) = \sum_{i=1}^{N} A_i \sin(\omega_i t + \psi_i), \qquad
\omega_i = \frac{2\pi}{T_i}$$

Dados de entrada (elevação do mar $\zeta_a(t)$ e movimento de heave $\zeta_{\text{heave}}(t)$).

In [ ]:
import random

t0 = 0
tf = 1 * 60 * 60
Nt = 10 * N
t = np.linspace(t0, tf, Nt)

random.seed(10)
y = np.zeros((N, Nt))
yh = np.zeros((N, Nt))

for ii, fi, ai, hi in zip(range(N), f, zeta_a, zeta_heave):
    phi = random.uniform(0, 2 * np.pi)
    y[ii, :] = ai * np.sin(2 * np.pi * fi * t + phi)
    yh[ii, :] = np.abs(hi) * np.sin(2 * np.pi * fi * t + phi + np.angle(hi))

ys = np.sum(y, 0)
yhs = np.sum(yh, 0)

In [ ]:
plt.figure(figsize=(10, 4), dpi=100, facecolor='w', edgecolor='k')
plt.subplot(2, 1, 1)
plt.plot(t, ys)
plt.ylabel('zeta_a(t) [m]')
plt.xlabel('t [s]')
plt.title(f'Série Temporal da Onda (Hs={hs:.2f} m / Tp={tp:.1f} s)')
plt.subplot(2, 1, 2)
plt.plot(t, yhs)
plt.ylabel('zeta_heave(t) [m]')
plt.xlabel('t [s]')
plt.xlim((200, 500))
plt.grid()
plt.tight_layout()
plt.show()

In [ ]:
Hs_mar = 4 * np.std(ys)
Hs_navio = 4 * np.std(yhs)
print(f'Hs_mar   = {Hs_mar:.3f} m')
print(f'Hs_navio = {Hs_navio:.3f} m')

## 6. Análise FFT

### 6.1 FFT da Onda

In [ ]:
from scipy.fft import fft
from scipy.signal import welch

yf = fft(ys)
N1 = len(yf)

Delta_f = 1 / (tf - t0)
f1 = np.arange(N1) * Delta_f

yf = 2 * yf[: N1 // 2]
w1 = 2 * pi * f1[: N1 // 2]

Delta_w = 2 * pi * Delta_f
zeta_a1 = np.abs(yf) / N1
Sw1 = (zeta_a1**2) / (2 * Delta_w)

f_welch, Sf_welch = welch(ys, fs=1 / (t[1] - t[0]), nperseg=int(len(ys) / 10))
w_welch = f_welch * (2 * pi)
Sw_welch = Sf_welch / (2 * pi)

plt.figure(figsize=(10, 4), dpi=100, facecolor='w', edgecolor='k')
plt.plot(w1, Sw1)
plt.plot(w, Sw, 'r')
plt.plot(w_welch, Sw_welch, 'y')
plt.ylabel('S(w) [m²·s]')
plt.xlabel('w [rad/s]')
plt.xlim((0, 2))
plt.legend(('FFT', 'Espectro Teórico', 'Welch'))
plt.grid()
plt.show()

In [ ]:
Hs_welch = 4 * np.sqrt(np.trapezoid(Sw_welch, w_welch))
Hs_welch

### 6.2 Estatísticas de Ondas

**$H_s$:** $H_s = 4\sigma$, $\;\sigma = \sqrt{m_0}$

**$T_z$:** $T_z = 2\pi\sqrt{m_0/m_2}$

**Momentos espectrais:** $m_n = \int_0^\infty \omega^n S(\omega)\,d\omega$

In [ ]:
m0 = np.trapezoid(Sw1, w1)
m1 = np.trapezoid((w1**1) * Sw1, w1)
m2 = np.trapezoid((w1**2) * Sw1, w1)
m4 = np.trapezoid((w1**4) * Sw1, w1)

Hs = 4 * np.sqrt(m0)
Tz = 2 * np.pi * np.sqrt(m0 / m2)
Tc = 2 * np.pi * (m0 / m1)

print(f'Hs = {Hs:.4f} m')
print(f'Tz = {Tz:.4f} s')
print(f'Tc = {Tc:.4f} s')

### 6.3 FFT do Movimento de Heave

In [ ]:
yhf = fft(yhs)
N1 = len(yhf)

yhf = 2 * yhf[: N1 // 2]
zeta_heave1 = np.abs(yhf) / N1
S_heave_w1 = (zeta_heave1**2) / (2 * Delta_w)

plt.figure(figsize=(10, 4), dpi=100, facecolor='w', edgecolor='k')
plt.plot(w1, np.abs(S_heave_w1))
plt.plot(w, np.abs(S_heave_w))
plt.ylabel('S_heave(w) [m²·s]')
plt.xlabel('w [rad/s]')
plt.xlim((0, 2))
plt.grid()
plt.show()

In [ ]:
m0 = np.trapezoid(np.abs(S_heave_w1), w1)
m2 = np.trapezoid((w1**2) * np.abs(S_heave_w1), w1)

Hs = 4 * np.sqrt(m0)
Tz = 2 * np.pi * np.sqrt(m0 / m2)

print(f'Hs = {Hs:.2f} m')
print(f'Tz = {Tz:.2f} s')

## 7. RAO Estimado via FFT

$$S_{\text{heave}}(\omega) = [\text{RAO}]^2 \cdot S_{\text{mar}}(\omega)
\qquad \Rightarrow \qquad
\text{RAO}(\omega) = \sqrt{\frac{S_{\text{heave}}(\omega)}{S_{\text{mar}}(\omega)}}$$

In [ ]:
RAO = np.sqrt(S_heave_w1 / Sw1)

plt.figure(figsize=(10, 6), dpi=100, facecolor='w', edgecolor='k')
plt.subplot(2, 1, 1)
plt.plot(w1, Sw1)
plt.plot(w, Sw)
plt.ylabel('S(w) [m²·s]')
plt.xlabel('w [rad/s]')
plt.xlim((0, 2))
plt.grid()
plt.subplot(2, 1, 2)
plt.plot(w1, RAO)
plt.plot(w, np.abs(rao_interpolado))
plt.xlim((0, 2))
plt.ylim((0, 2))
plt.ylabel('RAO(w) [m/m]')
plt.xlabel('w [rad/s]')
plt.grid()
plt.tight_layout()
plt.show()